# 📈 Portfolio Optimiser — Efficient Frontier & Sharpe Ratio

**Author:** Athu  
**Project:** Markowitz Mean-Variance Optimisation | Multi-Asset (ASX + Global)

---

## What This Notebook Covers

This notebook implements **Modern Portfolio Theory (Markowitz mean-variance optimisation)** — the foundation of quantitative asset allocation used by asset managers and quant teams.

| Step | What It Does | Industry Use |
|------|--------------|--------------|
| **Return & Covariance Estimation** | Annualised expected returns and risk | Input to any allocation model |
| **Random Portfolio Simulation** | Maps the feasible risk-return space | Intuition for the opportunity set |
| **Efficient Frontier** | Min-variance portfolio for each target return | Strategic asset allocation |
| **Max-Sharpe (Tangency) Portfolio** | Best risk-adjusted return | Optimal risky portfolio |
| **Minimum-Variance Portfolio** | Lowest possible risk | Defensive allocation |
| **Capital Market Line** | Risk-free asset + tangency portfolio | CAPM / portfolio selection |

### Portfolio Universe (ASX + Global)
- **Australian equity:** STW.AX (ASX 200)
- **Global equity:** SPY (S&P 500), EFA (Int'l Developed), EEM (Emerging Markets)
- **Bonds:** TLT (US Long Treasury)
- **Commodity:** GLD (Gold)


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

RF = 0.04          # annual risk-free rate (RBA cash ~ 4%)
TRADING_DAYS = 252
print("✅ Libraries loaded")

## 2. Data Acquisition

We pull **5 years of daily prices** for a diversified ASX + global universe via `yfinance`. A longer window gives more stable covariance estimates for optimisation.


In [ ]:
tickers = {
    'STW.AX': 'ASX 200',
    'SPY':    'S&P 500',
    'EFA':    "Int'l Developed",
    'EEM':    'Emerging Mkts',
    'TLT':    'US Long Treasury',
    'GLD':    'Gold',
}

prices = yf.download(list(tickers.keys()), period='5y', auto_adjust=True)['Close']
prices = prices.dropna()
prices.columns = [tickers.get(c, c) for c in prices.columns]

print(f"Date range: {prices.index[0]:%Y-%m-%d} to {prices.index[-1]:%Y-%m-%d}")
print(f"Trading days: {len(prices)}  |  Assets: {prices.shape[1]}")
prices.tail()

## 3. Returns, Expected Returns & Covariance

We use **log returns** and annualise the mean and covariance matrix. These are the only inputs Markowitz optimisation needs.

$$\mu_{ann} = \bar{r}_{daily} \times 252 \qquad \Sigma_{ann} = \Sigma_{daily} \times 252$$


In [ ]:
returns = np.log(prices / prices.shift(1)).dropna()

mean_returns = returns.mean() * TRADING_DAYS          # annualised expected return
cov_matrix   = returns.cov() * TRADING_DAYS           # annualised covariance
assets = returns.columns.tolist()
n = len(assets)

summary = pd.DataFrame({
    'Ann. Return (%)': (mean_returns * 100).round(2),
    'Ann. Volatility (%)': (np.sqrt(np.diag(cov_matrix)) * 100).round(2),
    'Sharpe (Rf=4%)': ((mean_returns - RF) / np.sqrt(np.diag(cov_matrix))).round(3),
})
print("Individual asset profile:\n")
summary

## 4. Portfolio Performance Function

For any weight vector **w**, we compute annualised return, volatility, and Sharpe ratio.

$$R_p = w^T \mu \qquad \sigma_p = \sqrt{w^T \Sigma\, w} \qquad Sharpe = \frac{R_p - R_f}{\sigma_p}$$


In [ ]:
def portfolio_performance(w, mean_returns=mean_returns, cov_matrix=cov_matrix, rf=RF):
    w = np.array(w)
    ret = w @ mean_returns
    vol = np.sqrt(w @ cov_matrix.values @ w)
    sharpe = (ret - rf) / vol
    return ret, vol, sharpe

## 5. Random Portfolio Simulation

Before optimising, we **simulate thousands of random long-only portfolios** to visualise the feasible risk-return space. The upper-left edge of this cloud is the efficient frontier.


In [ ]:
np.random.seed(42)
n_portfolios = 20_000
results = np.zeros((3, n_portfolios))
weights_record = []

for i in range(n_portfolios):
    w = np.random.random(n)
    w /= w.sum()                      # long-only, fully invested
    weights_record.append(w)
    r, v, s = portfolio_performance(w)
    results[0, i], results[1, i], results[2, i] = v, r, s

sim = pd.DataFrame({'Volatility': results[0], 'Return': results[1], 'Sharpe': results[2]})
print(f"Simulated {n_portfolios:,} random portfolios")
print(f"Best random Sharpe: {sim['Sharpe'].max():.3f}")

## 6. Optimisation — Max Sharpe & Minimum Variance

We use `scipy.optimize.minimize` (SLSQP) with constraints:
- **Fully invested:** weights sum to 1
- **Long-only:** each weight between 0 and 1

The **max-Sharpe (tangency)** portfolio maximises risk-adjusted return; the **minimum-variance** portfolio minimises total risk.


In [ ]:
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
bounds = tuple((0, 1) for _ in range(n))
x0 = np.repeat(1 / n, n)

# Max Sharpe: minimise the negative Sharpe ratio
def neg_sharpe(w):
    return -portfolio_performance(w)[2]

# Min variance: minimise portfolio variance
def portfolio_variance(w):
    return w @ cov_matrix.values @ w

opt_sharpe = minimize(neg_sharpe, x0, method='SLSQP', bounds=bounds, constraints=constraints)
opt_minvar = minimize(portfolio_variance, x0, method='SLSQP', bounds=bounds, constraints=constraints)

w_sharpe = opt_sharpe.x
w_minvar = opt_minvar.x

for name, w in [('Max-Sharpe (Tangency)', w_sharpe), ('Minimum-Variance', w_minvar)]:
    r, v, s = portfolio_performance(w)
    print(f"\n{name} portfolio:")
    print(f"  Return {r:.2%} | Volatility {v:.2%} | Sharpe {s:.3f}")
    for a, wt in zip(assets, w):
        print(f"    {a:18s} {wt:6.1%}")

## 7. Efficient Frontier

For a grid of target returns, we find the **minimum-variance portfolio** achieving each target. The resulting curve is the efficient frontier — the set of portfolios offering the best return for each level of risk.


In [ ]:
def efficient_frontier(target_returns):
    vols = []
    for target in target_returns:
        cons = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w, t=target: w @ mean_returns - t},
        )
        res = minimize(portfolio_variance, x0, method='SLSQP', bounds=bounds, constraints=cons)
        vols.append(np.sqrt(res.fun))
    return np.array(vols)

r_min = portfolio_performance(w_minvar)[0]
r_max = mean_returns.max()
target_returns = np.linspace(r_min, r_max, 50)
frontier_vols = efficient_frontier(target_returns)
print(f"Computed {len(target_returns)} efficient-frontier portfolios")

## 8. Visualisation — The Efficient Frontier

We plot the random portfolio cloud (coloured by Sharpe), the efficient frontier, the **Capital Market Line** (risk-free asset + tangency portfolio), and mark the optimal portfolios.


In [ ]:
r_s, v_s, s_s = portfolio_performance(w_sharpe)
r_m, v_m, _   = portfolio_performance(w_minvar)

fig, ax = plt.subplots(figsize=(13, 8))
sc = ax.scatter(sim['Volatility'], sim['Return'], c=sim['Sharpe'],
                cmap='viridis', s=6, alpha=0.4)
plt.colorbar(sc, label='Sharpe Ratio')

ax.plot(frontier_vols, target_returns, 'k-', linewidth=2.5, label='Efficient Frontier')

# Capital Market Line
cml_x = np.linspace(0, sim['Volatility'].max(), 50)
ax.plot(cml_x, RF + s_s * cml_x, 'r--', linewidth=1.8, label=f'Capital Market Line (slope={s_s:.2f})')

ax.scatter(v_s, r_s, marker='*', color='red',   s=500, edgecolors='black', label=f'Max Sharpe ({s_s:.2f})', zorder=5)
ax.scatter(v_m, r_m, marker='*', color='blue',  s=500, edgecolors='black', label='Min Variance', zorder=5)

ax.set_xlabel('Annualised Volatility')
ax.set_ylabel('Annualised Return')
ax.set_title('Efficient Frontier — Markowitz Mean-Variance Optimisation', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 9. Optimal Weights — Summary

In [ ]:
alloc = pd.DataFrame({
    'Max-Sharpe (%)': (w_sharpe * 100).round(1),
    'Min-Variance (%)': (w_minvar * 100).round(1),
    'Equal-Weight (%)': (np.repeat(1/n, n) * 100).round(1),
}, index=assets)

print("=" * 60)
print("        OPTIMAL PORTFOLIO ALLOCATIONS")
print("=" * 60)
print(alloc.to_string())

print("\nPerformance comparison:")
for name, w in [('Max-Sharpe', w_sharpe), ('Min-Variance', w_minvar), ('Equal-Weight', np.repeat(1/n, n))]:
    r, v, s = portfolio_performance(w)
    print(f"  {name:14s}  Return {r:6.2%} | Vol {v:6.2%} | Sharpe {s:5.3f}")

fig, ax = plt.subplots(figsize=(11, 6))
alloc[['Max-Sharpe (%)', 'Min-Variance (%)']].plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4'])
ax.set_title('Optimal Weights by Strategy', fontsize=13, fontweight='bold')
ax.set_ylabel('Weight (%)')
ax.set_xticklabels(assets, rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 🚀 Next Steps & Interview Talking Points

### Extensions
- [ ] **Add short-selling** — relax the long-only bound to allow negative weights
- [ ] **Weight constraints** — cap any single asset (e.g. ≤ 30%) for realistic mandates
- [ ] **Shrinkage covariance** — Ledoit-Wolf estimator for more stable inputs
- [ ] **Black-Litterman** — blend market equilibrium with your own views
- [ ] **Backtest** — compare optimal vs equal-weight out-of-sample

### Interview Talking Points
1. "I implemented Markowitz mean-variance optimisation with SLSQP under fully-invested, long-only constraints."
2. "I located the tangency portfolio by maximising the Sharpe ratio and traced the full efficient frontier."
3. "I visualised the feasible set with 20,000 simulated portfolios and overlaid the Capital Market Line."
4. "Key limitation: mean-variance is highly sensitive to expected-return estimates — which motivates shrinkage and Black-Litterman."
